In [1]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

class OnlineIncrementalLearning:
    def __init__(self, num_clusters=3, threshold=0.5):
        self.num_clusters = num_clusters
        self.threshold = threshold
        self.previous_centroids = None

    def online_learning(self, data_batch):
        # Perform clustering on the incoming data batch
        kmeans = KMeans(n_clusters=self.num_clusters)
        kmeans.fit(data_batch)
        centroids = kmeans.cluster_centers_

        # Calculate distance between previous and current centroids
        if self.previous_centroids is not None:
            distances = pairwise_distances(centroids, self.previous_centroids)
            concept_drift_detected = np.max(distances) > self.threshold
            if concept_drift_detected:
                print("Concept drift detected!")
            else:
                print("No significant concept drift.")
        else:
            print("Initial clustering complete.")

        self.previous_centroids = centroids

    def incremental_learning(self, data_batch):
        # Fine-tuning centroids incrementally
        for point in data_batch:
            closest_centroid = np.argmin(
                np.linalg.norm(self.previous_centroids - point, axis=1)
            )
            # Update closest centroid incrementally
            self.previous_centroids[closest_centroid] += 0.1 * (point - self.previous_centroids[closest_centroid])
            print(f"Updated centroid {closest_centroid} to {self.previous_centroids[closest_centroid]}")

    def decremental_learning(self, data_batch):
        # Forget outdated centroids based on inactivity
        active_centroids = np.array([centroid for centroid in self.previous_centroids if self._is_active(centroid, data_batch)])
        self.previous_centroids = active_centroids
        print("Decremental learning applied. Updated centroids:", self.previous_centroids)

    def _is_active(self, centroid, data_batch):
        # A heuristic to determine activity (thresholding proximity to batch points)
        for point in data_batch:
            if np.linalg.norm(centroid - point) < self.threshold:
                return True
        return False

# Example Usage
# Simulated batches of data
batch_1 = np.random.rand(100, 2)  # 100 points in 2D space
batch_2 = np.random.rand(100, 2)

# Initialize the learning model
online_model = OnlineIncrementalLearning()

# Process batches
online_model.online_learning(batch_1)
online_model.incremental_learning(batch_2)
online_model.decremental_learning(batch_2)



Initial clustering complete.
Updated centroid 2 to [0.71098699 0.17131164]
Updated centroid 0 to [0.70569658 0.74152097]
Updated centroid 1 to [0.15910106 0.37568817]
Updated centroid 1 to [0.16625215 0.39278069]
Updated centroid 0 to [0.71565558 0.72857664]
Updated centroid 0 to [0.7406823  0.72154646]
Updated centroid 2 to [0.72324924 0.17523877]
Updated centroid 0 to [0.75697061 0.7097299 ]
Updated centroid 1 to [0.1717292  0.43240294]
Updated centroid 0 to [0.74838366 0.69345844]
Updated centroid 1 to [0.15654654 0.39529837]
Updated centroid 2 to [0.73809137 0.17341584]
Updated centroid 0 to [0.72088814 0.70530246]
Updated centroid 0 to [0.7288169  0.71187091]
Updated centroid 0 to [0.70286127 0.70441382]
Updated centroid 1 to [0.14527925 0.43122869]
Updated centroid 0 to [0.67988259 0.6903379 ]
Updated centroid 2 to [0.75380608 0.17338413]
Updated centroid 1 to [0.14182274 0.43331731]
Updated centroid 0 to [0.70702914 0.68144027]
Updated centroid 0 to [0.71306004 0.66944612]
Updat

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# Load Dataset
traffic_data = pd.read_csv('traffic_with_sentiment_dataset.csv')

# Apply KMeans Clustering
num_clusters = 3  # Number of clusters
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
traffic_data['Cluster'] = kmeans.fit_predict(traffic_data[['Vehicle Count', 'Congestion Indicator']])

# Add 'Spatiotemporal Flow' Column
def calculate_spatiotemporal_flow(row):
    # Generate synthetic values based on vehicle count and congestion level
    return row['Vehicle Count'] * (1 - row['Congestion Indicator']) * np.random.uniform(0.8, 1.2)

traffic_data['Spatiotemporal Flow'] = traffic_data.apply(calculate_spatiotemporal_flow, axis=1)

# Analyze Clusters
for cluster_id in range(num_clusters):
    cluster = traffic_data[traffic_data['Cluster'] == cluster_id]
    print(f"Cluster {cluster_id}:")
    print(f"  Average Vehicle Count: {cluster['Vehicle Count'].mean()}")
    print(f"  Average Congestion: {cluster['Congestion Indicator'].mean()}")
    print(f"  High Traffic Areas: {cluster['Location ID'].unique()}")
    print(f"  Average Spatiotemporal Flow: {cluster['Spatiotemporal Flow'].mean()}")



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

# Check if 'Weather Conditions' column exists and encode it
if 'Weather Conditions' in traffic_data.columns:
    encoder = LabelEncoder()
    traffic_data['Weather Conditions'] = encoder.fit_transform(traffic_data['Weather Conditions'])

# Prepare Data for Supervised Learning
features = ['Cluster', 'Vehicle Count', 'Congestion Indicator']
if 'Weather Conditions' in traffic_data.columns:
    features.append('Weather Conditions')  # Add if available

X = traffic_data.loc[:, features]  # Use .loc[] to avoid SettingWithCopyWarning

# Ensure 'Spatiotemporal Flow' column exists
if 'Spatiotemporal Flow' not in traffic_data.columns:
    print("Error: 'Spatiotemporal Flow' column is missing from the dataset!")
else:
    y = traffic_data['Spatiotemporal Flow']  # Target variable

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train Supervised Model
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate Model
    y_pred = model.predict(X_test)
    print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred)}")


In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Load Dataset
file_path = 'traffic_with_sentiment_dataset.csv'  # Update file path as needed
traffic_data = pd.read_csv(file_path)

# Step 2: Add 'Spatiotemporal Flow' Column
def calculate_spatiotemporal_flow(row):
    # Generate synthetic 'Spatiotemporal Flow' based on vehicle count and congestion
    return row['Vehicle Count'] * (1 - row['Congestion Indicator']) * np.random.uniform(0.8, 1.2)

traffic_data['Spatiotemporal Flow'] = traffic_data.apply(calculate_spatiotemporal_flow, axis=1)

# Step 3: Feature Engineering
def feature_engineering(data):
    # Encode 'Weather Conditions' if available
    if 'Weather Conditions' in data.columns:
        label_encoder = LabelEncoder()
        data['Weather Conditions'] = label_encoder.fit_transform(data['Weather Conditions'])
    
    # Add synthetic 'Event Indicator' column if not already present
    if 'Event Indicator' not in data.columns:
        data['Event Indicator'] = np.random.choice(['Accident', 'Roadwork', 'None'], size=len(data), p=[0.2, 0.3, 0.5])
        event_encoder = LabelEncoder()
        data['Event Indicator'] = event_encoder.fit_transform(data['Event Indicator'])
    
    # Add Lagged Features
    data['Lagged Vehicle Count'] = data['Vehicle Count'].shift(1).fillna(data['Vehicle Count'].mean())
    data['Lagged Congestion Indicator'] = data['Congestion Indicator'].shift(1).fillna(data['Congestion Indicator'].mean())
    return data

traffic_data = feature_engineering(traffic_data)

# Step 4: Apply Clustering to Create 'Cluster' Column
def apply_clustering(data, num_clusters=3):
    clustering_features = ['Vehicle Count', 'Congestion Indicator', 'Lagged Vehicle Count', 'Lagged Congestion Indicator']
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    data['Cluster'] = kmeans.fit_predict(data[clustering_features])
    return data

traffic_data = apply_clustering(traffic_data)

# Step 5: Data Preprocessing
def data_preprocessing(data):
    # Separate numeric and non-numeric columns
    numeric_columns = data.select_dtypes(include=['number']).columns
    non_numeric_columns = data.select_dtypes(exclude=['number']).columns
    
    # Handle missing values for numeric and non-numeric columns
    data[numeric_columns] = data[numeric_columns].fillna(data[numeric_columns].mean())
    data[non_numeric_columns] = data[non_numeric_columns].fillna("Unknown")
    
    # Normalize numeric columns
    scaler = StandardScaler()
    data[numeric_columns] = scaler.fit_transform(data[numeric_columns])
    
    return data

traffic_data = data_preprocessing(traffic_data)

# Step 6: Prepare Data for Modeling
X = traffic_data[['Cluster', 'Vehicle Count', 'Congestion Indicator', 'Lagged Vehicle Count', 'Lagged Congestion Indicator']]
y = traffic_data['Spatiotemporal Flow']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 7: Model Tuning (Random Forest Regressor)
def tune_random_forest(X_train, y_train, X_test, y_test):
    model = RandomForestRegressor(n_estimators=150, max_depth=20, min_samples_split=5, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate Model
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"Random Forest MSE: {mse}, R2 Score: {r2}")
    return model

rf_model = tune_random_forest(X_train, y_train, X_test, y_test)

# Step 8: Cross-Validation
def cross_validate_model(model, X, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
    print(f"Cross-Validation Scores: {scores}")
    print(f"Average Cross-Validation MSE: {np.mean(-scores)}")

cross_validate_model(rf_model, X, y)

# Step 9: Analyze Residuals
def analyze_residuals(y_test, y_pred):
    residuals = y_test - y_pred
    plt.figure(figsize=(8, 6))
    plt.scatter(y_test, residuals)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Actual Values')
    plt.ylabel('Residuals')
    plt.title('Residual Analysis')
    plt.show()

y_pred = rf_model.predict(X_test)
analyze_residuals(y_test, y_pred)

# Step 10: Evaluate Data Quality
def evaluate_data_quality(data):
    print("Missing Values:\n", data.isnull().sum())
    
    # Select only numeric columns for correlation calculation
    numeric_data = data.select_dtypes(include=['number'])
    
    # Compute Correlations
    print("Feature Correlations:\n", numeric_data.corr())
    
    # Plot Heatmap for Numeric Data
    sns.heatmap(numeric_data.corr(), annot=True, cmap="coolwarm")
    plt.title("Correlation Heatmap")
    plt.show()

evaluate_data_quality(traffic_data)


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Plot Heatmap of Traffic Density
pivot = traffic_data.pivot_table(index='Timestamp', columns='Location ID', values='Vehicle Count', aggfunc='mean')
plt.figure(figsize=(12, 6))
sns.heatmap(pivot, cmap='coolwarm')
plt.title('Traffic Density Heatmap')
plt.xlabel('Location ID')
plt.ylabel('Timestamp')
plt.show()

# Cluster Scatter Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=traffic_data, x='Longitude', y='Latitude', hue='Cluster', palette='viridis')
plt.title('Traffic Clusters by Location')
plt.show()
